# Customize the Vulnerability Analysis Pipeline


This notebook continues from `1_Deploy_CVE.ipynb` and focuses on customizing the current README workflow. The pipeline input is a `PipelineInput` JSON document: one or more CVE/GHSA identifiers, `image_name`, `image_tag`, optional rootfs paths, cache controls, and output settings.

The previous SBOM/source-repository payload format is no longer the primary workflow input. Package verification is now filesystem-based, and automatic image extraction is handled by the orchestrator when `filesystem_path` is omitted.

**Cell classifications:** every code cell is classified in notebook metadata as `setup-only`, `executable`, or `illustrative`. Setup-only cells initialize state, executable cells are expected to run as written in a configured environment (some are explicitly opt-in), and illustrative cells only print templates or cleanup guidance.


# Table of Contents

>[Prerequisites and Setup](#Prerequisites-and-Setup)  
>[Customize PipelineInput](#Customize-PipelineInput)  
>[Use a Pre-Extracted Rootfs](#Use-a-Pre-Extracted-Rootfs)  
>[Run Custom Inputs](#Run-Custom-Inputs)  
>[Customize Configuration](#Customize-Configuration)  
>[Run Individual Agents](#Run-Individual-Agents)  
>[Benchmarking](#Benchmarking)  
>[Wrapping Up](#Wrapping-Up)


## Prerequisites and Setup

Make sure the repository is installed locally with `uv sync`, or that the Docker Compose deployment from the first notebook is still running. All paths below are relative to the repository root.

Run this notebook from top to bottom. Earlier cells create `/tmp/custom_pipeline_input.json`; the HTTP cell creates `/tmp/custom_generate_response.json`; later standalone-agent examples consume both. HTTP examples require the service deployed by `1_Deploy_CVE.ipynb`. After a kernel restart, rerun setup and producer cells rather than trusting old `/tmp` artifacts.

Both `NVIDIA_API_KEY` and `TAVILY_API_KEY` are required for a complete uncached run. Supply them through a project-root `.env`, exported environment variables, or notebook/orchestrator secret injection, and never commit real credentials.


In [ ]:
from pathlib import Path
import httpx
import json
import os
import re
import shlex
import subprocess

from vuln_analysis.semantic_failure import require_successful_pipeline_result


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "vuln_analysis").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the vulnerability-analysis repository.")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
os.environ["REPO_ROOT"] = str(REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")


def extract_pipeline_result(text: str) -> dict:
    decoder = json.JSONDecoder()
    matches = []
    for match in re.finditer(r"\{", text):
        try:
            parsed, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict) and {"image_name", "image_tag", "results"} <= set(parsed):
            matches.append(parsed)
    if not matches:
        raise RuntimeError("Could not find a PipelineResult in NAT output.")
    return matches[-1]


## Customize PipelineInput

The full workflow accepts the fields below.

| Field | Description |
|---|---|
| `vulns` | List of vulnerability entries. Each entry has `vuln_id` and optional `package_hint` / `version_hint`. |
| `image_name` | Container repository/name without the tag. |
| `image_tag` | Container tag. Required for automatic and pre-extracted inputs. |
| `filesystem_path` | Optional path to a pre-extracted rootfs. If omitted, the pipeline pulls and extracts `image_name:image_tag` with `crane`. |
| `image_config_path` | Optional path to image config JSON; auto-populated for automatic extraction. |
| `skip_web_research` | Set `true` to skip container web research, useful for private images. |
| `output_dir` | Root directory for reports and VEX results. |
| `cache` | Set `false` to disable stage caching for this run. |
| `cache_cve_researcher`, `cache_container_analyzer`, `cache_exploitability_analyzer`, `cache_vex_categorizer` | Optional per-agent cache overrides. |

> **Note:** Relative `output_dir` values are resolved from the repository root, so the examples store results within the cloned repository, such as `<repo>/outputs/custom-morpheus`.


Create a single-CVE input that uses automatic rootfs extraction.


In [ ]:
custom_input = {
    "vulns": [{"vuln_id": "GHSA-53q9-r3pm-6pq6"}],
    "image_name": "nvcr.io/nvidia/morpheus/morpheus",
    "image_tag": "25.06-runtime",
    "output_dir": "outputs/custom-morpheus",
    "cache": True,
}

custom_input_path = Path("/tmp/custom_pipeline_input.json")
custom_input_path.parent.mkdir(parents=True, exist_ok=True)
custom_input_path.write_text(json.dumps(custom_input, indent=2) + "\n")
print(custom_input_path)
print(json.dumps(custom_input, indent=2))


Create a multi-CVE input from the three reviewed Morpheus eval cases. The Container Analyzer runs once, while CVE Researcher, Exploitability Analyzer, and VEX Categorizer fan out per vulnerability.


In [ ]:
multi_cve_input = {
    "vulns": [
        {"vuln_id": "GHSA-53q9-r3pm-6pq6"},
        {"vuln_id": "GHSA-ph84-rcj2-fxxm"},
        {"vuln_id": "CVE-2024-9287"},
    ],
    "image_name": "nvcr.io/nvidia/morpheus/morpheus",
    "image_tag": "25.06-runtime",
    "output_dir": "outputs/custom-morpheus-multi",
    "cache": True,
}

multi_input_path = Path("/tmp/custom_multi_cve_input.json")
multi_input_path.write_text(json.dumps(multi_cve_input, indent=2) + "\n")
print(multi_input_path)
print(json.dumps(multi_cve_input, indent=2))


**Illustrative cell:** for private images, set `skip_web_research` to `true`. The `registry.example.com` values below are placeholders and are not expected to run as written. The filesystem analysis still runs for a real accessible image, but the Container Analyzer skips public registry/vendor searches.


In [ ]:
private_image_input = {
    "vulns": [{"vuln_id": "GHSA-53q9-r3pm-6pq6"}],
    "image_name": "registry.example.com/team/private-app",
    "image_tag": "2026.06.03",
    "skip_web_research": True,
    "output_dir": "outputs/private-app",
    "cache": False,
}

print(json.dumps(private_image_input, indent=2))


## Use a Pre-Extracted Rootfs

Automatic extraction is recommended for most runs. The opt-in cell below runs the standalone helper, parses its reported paths, and writes a connected `/tmp/pre_extracted_pipeline_input.json`. Set `RUN_PRE_EXTRACTION = True` only when you want to pull the example image; extraction can take several minutes and may require registry authentication.


In [ ]:
RUN_PRE_EXTRACTION = False
pre_extracted_input_path = Path("/tmp/pre_extracted_pipeline_input.json")
pre_extracted_input_path.unlink(missing_ok=True)

extract_command = [
    "uv", "run", "python", "scripts/extract_image.py",
    "nvcr.io/nvidia/morpheus/morpheus:25.06-runtime",
    "/tmp/image-cache",
]
print("$ " + shlex.join(extract_command))

if RUN_PRE_EXTRACTION:
    completed = subprocess.run(extract_command, check=True, capture_output=True, text=True)
    print(completed.stdout)
    extraction = {}
    for line in completed.stdout.splitlines():
        key, separator, value = line.partition(":")
        if separator:
            extraction[key.strip()] = value.strip()
    required_paths = {"rootfs_path", "config_path"}
    missing_paths = required_paths - extraction.keys()
    if missing_paths:
        raise RuntimeError(f"Extraction helper did not report: {sorted(missing_paths)}")
    for key in required_paths:
        if not Path(extraction[key]).exists():
            raise RuntimeError(f"Extraction helper reported a missing {key}: {extraction[key]}")

    pre_extracted_input = {
        "vulns": [{"vuln_id": "GHSA-53q9-r3pm-6pq6"}],
        "image_name": "nvcr.io/nvidia/morpheus/morpheus",
        "image_tag": "25.06-runtime",
        "filesystem_path": extraction["rootfs_path"],
        "image_config_path": extraction["config_path"],
        "output_dir": "outputs/pre-extracted-morpheus",
    }
    pre_extracted_input_path.write_text(json.dumps(pre_extracted_input, indent=2) + "\n")
    print(f"Wrote {pre_extracted_input_path}")
else:
    print("Optional extraction was not run; no pre-extracted input file was created.")


The producer cell removes stale output before it starts. When extraction is enabled and succeeds, inspect the generated, runnable input below.


In [ ]:
if pre_extracted_input_path.exists():
    print(pre_extracted_input_path.read_text())
else:
    print("Set RUN_PRE_EXTRACTION = True and rerun the producer cell first.")


## Run Custom Inputs

Run the input produced earlier locally with `nat run`. The cell requires a zero process exit and validates top-level/per-vulnerability success, every required stage and report path, VEX fields, and VEX JSON paths.


In [ ]:
# This can take several minutes on first run because the rootfs and reports are cached.
custom_run_command = [
    "uv", "run", "nat", "run",
    "--config_file=configs/config.yml",
    "--input_file=/tmp/custom_pipeline_input.json",
]
custom_run = subprocess.run(
    custom_run_command,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(custom_run.stdout)
custom_run.check_returncode()
custom_local_result = require_successful_pipeline_result(extract_pipeline_result(custom_run.stdout))
print(f"Validated {len(custom_local_result.results)} successful local result(s).")


If the HTTP server from the deploy notebook is running, submit the same input to `/generate`. The response is saved for standalone-input generation only after the cell rejects `workflow_error`, HTTP semantic failure, incomplete stages, and missing report/result fields.


In [ ]:
custom_payload = json.loads(Path("/tmp/custom_pipeline_input.json").read_text())
custom_response = httpx.post(
    "http://localhost:26466/generate",
    json=custom_payload,
    timeout=3600,
)
try:
    custom_response_payload = custom_response.json()
except ValueError as exc:
    raise RuntimeError(f"Custom API returned non-JSON HTTP {custom_response.status_code}") from exc
custom_response_path = Path("/tmp/custom_generate_response.json")
custom_response_path.write_text(json.dumps(custom_response_payload, indent=2) + "\n")
print(custom_response_path.read_text()[:12000])
custom_http_result = require_successful_pipeline_result(custom_response_payload)
custom_response.raise_for_status()
print(f"Validated {len(custom_http_result.results)} successful HTTP result(s).")


## Customize Configuration

The default config is `configs/config.yml`, a repo-root symlink to `src/vuln_analysis/configs/config.yml`. Key defaults from the README:

| Setting | Default |
|---|---|
| Default LLM model | `nvidia/nemotron-3-ultra-550b-a55b` |
| Stage timeout | `600` seconds |
| Max concurrency | `null` (unlimited) |
| LLM rate limit | `null` (no limit) |
| Output directory | `outputs` |

Model names are controlled with environment variables: `DEFAULT_MODEL_NAME`, `CVE_RESEARCHER_MODEL`, `CONTAINER_ANALYZER_MODEL`, `EXPLOITABILITY_ANALYZER_MODEL`, and `VEX_CATEGORIZER_MODEL`.


In [ ]:
%%bash
sed -n '1,80p' configs/config.yml
sed -n '/^workflow:/,$p' configs/config.yml


Create a custom config that caps concurrent agent tasks and spaces LLM calls. Use these knobs only when your inference endpoint needs throttling.


In [ ]:
from pathlib import Path

base_config = Path("configs/config.yml").read_text()
rate_limited_config = base_config.replace("  max_concurrency: null", "  max_concurrency: 3")
rate_limited_config = rate_limited_config.replace("  llm_max_rate: null", "  llm_max_rate: 5")

custom_config_path = Path("/tmp/config-rate-limited.yml")
custom_config_path.parent.mkdir(parents=True, exist_ok=True)
custom_config_path.write_text(rate_limited_config)
print(custom_config_path)


In [ ]:
rate_limited_command = [
    "uv", "run", "nat", "run",
    "--config_file=/tmp/config-rate-limited.yml",
    "--input_file=/tmp/custom_pipeline_input.json",
]
rate_limited_run = subprocess.run(
    rate_limited_command,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(rate_limited_run.stdout)
rate_limited_run.check_returncode()
rate_limited_result = require_successful_pipeline_result(extract_pipeline_result(rate_limited_run.stdout))
print(f"Validated {len(rate_limited_result.results)} successful rate-limited result(s).")


### Environment Overrides

Update `.env` or export values in your shell before starting `nat run` / `nat serve`.

```env
NVIDIA_API_KEY=your-nvidia-api-key
NV_BASE_URL=https://integrate.api.nvidia.com/v1
TAVILY_API_KEY=your-tavily-api-key
DEFAULT_MODEL_NAME=nvidia/nemotron-3-ultra-550b-a55b
CVE_RESEARCHER_MODEL=nvidia/nemotron-3-ultra-550b-a55b
CONTAINER_ANALYZER_MODEL=nvidia/nemotron-3-ultra-550b-a55b
EXPLOITABILITY_ANALYZER_MODEL=nvidia/nemotron-3-ultra-550b-a55b
VEX_CATEGORIZER_MODEL=nvidia/nemotron-3-ultra-550b-a55b
CACHE_ENABLED=true
VULN_ANALYSIS_IMAGE_CACHE=.cache/vuln_analysis/images
```


## Run Individual Agents

Each agent can be run independently for targeted debugging. `data/examples/cve_researcher_input.json` is directly runnable. The checked-in downstream examples are schema illustrations because their report and rootfs paths depend on a real pipeline run.

After the earlier HTTP cell has produced `/tmp/custom_generate_response.json`, the next cell can generate connected downstream inputs from that result and `/tmp/custom_pipeline_input.json`. Leave the opt-in flag false when using **Run All** without a completed response.


In [ ]:
RUN_STANDALONE_INPUT_GENERATION = False
standalone_input_dir = Path("/tmp/standalone-inputs")
generate_command = [
    "uv", "run", "vuln-analysis", "generate-standalone-inputs",
    "--pipeline-input", "/tmp/custom_pipeline_input.json",
    "--pipeline-result", "/tmp/custom_generate_response.json",
    "--output-dir", str(standalone_input_dir),
]
print("$ " + shlex.join(generate_command))
if RUN_STANDALONE_INPUT_GENERATION:
    for required_path in (Path("/tmp/custom_pipeline_input.json"), Path("/tmp/custom_generate_response.json")):
        if not required_path.is_file():
            raise RuntimeError(f"Run its producer cell first; missing {required_path}")
    subprocess.run(generate_command, check=True)
else:
    print("Generation was not run. Complete the HTTP example, then enable the flag.")

standalone_commands = {
    "CVE Researcher": ["uv", "run", "nat", "run", "--config_file=configs/config-cve-researcher.yml", "--input_file=data/examples/cve_researcher_input.json"],
    "Container Analyzer": ["uv", "run", "nat", "run", "--config_file=configs/config-container-analyzer.yml", f"--input_file={standalone_input_dir / 'container_analyzer_input.json'}"],
    "Exploitability Analyzer": ["uv", "run", "nat", "run", "--config_file=configs/config-exploitability-analyzer.yml", f"--input_file={standalone_input_dir / 'exploitability_analyzer_input.json'}"],
    "VEX Categorizer": ["uv", "run", "nat", "run", "--config_file=configs/config-vex-categorizer.yml", f"--input_file={standalone_input_dir / 'vex_categorizer_input.json'}"],
}
for agent, command in standalone_commands.items():
    print(f"{agent}: {shlex.join(command)}")


## Benchmarking

Labelled datasets live under `data/eval_datasets/`, and evaluations run through `nat eval` with `configs/config-eval.yml`. The checked-in `morpheus_2506_eval_test.csv` is a three-case Morpheus runtime smoke/regression fixture: one `vulnerable` and two `code_not_reachable` cases. It is intentionally small and is not a statistically representative benchmark.


In [ ]:
RUN_EVALUATION = False
eval_dataset = Path("data/eval_datasets/morpheus_2506_eval_test.csv")
if not eval_dataset.is_file():
    raise RuntimeError(f"Evaluation dataset is missing: {eval_dataset}")

eval_command = [
    "uv", "run", "nat", "eval",
    "--config_file=configs/config-eval.yml",
    "--override", "eval.general.dataset.file_path", str(eval_dataset),
    "--deadline-seconds", "3600",
]
print("$ " + shlex.join(eval_command))
if RUN_EVALUATION:
    subprocess.run(eval_command, check=True)
else:
    print("Evaluation was not run. Set RUN_EVALUATION = True to execute this command.")


## Wrapping Up

The following cell is illustrative: its commands remain commented so **Run All** cannot delete reports or stop the deployment. Review and run only the command you intend.


In [ ]:
%%bash
# Stop the Docker Compose services from the repository root
# bash deploy/compose.sh down

# Preview application cache cleanup; add --yes only after reviewing targets
# vuln-analysis cache prune --all

# Disable caching for one input by setting "cache": false in PipelineInput
